# Evaluate a Google ADK weather agent

[Companion article](https://futureagi.com/blog/evaluate-google-adk-agents) · [Open in Colab](https://colab.research.google.com/github/future-agi/cookbooks/blob/main/google_adk_eval_loop/Google_ADK_Eval_Loop.ipynb)

Run a tool-using agent against a fixed weather report. Keep the tool call, the reply and its groundedness evaluation in one Observe trace, then check what you would allow through a CI gate. No weather service is needed.

## Install and authenticate

Use a fresh Python 3.12 kernel. The agent uses OpenAI through ADK’s LiteLLM adapter; the hosted judge and trace export use your Future AGI keys.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                'agent-learning-kit==0.3.7', 'traceai-google-adk==0.1.7', 'fi-instrumentation-otel==1.5.1', 'google-adk==2.7.1'], check=True)

In [ ]:
import os
from getpass import getpass

for name in ('FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY'):
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value.strip():
            raise ValueError(f"{name} is required.")
        os.environ[name] = value.strip()
        del value

## Instrument ADK and the evaluator

Run this setup once per kernel. Set `FI_BASE_URL` and `FI_TRACES_URL` before running it if you self-host the platform and collector separately.

In [ ]:
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from fi.evals import evaluate, Evaluator
from fi.evals.otel import enable_auto_enrichment
from traceai_google_adk import GoogleADKInstrumentor

provider = register(project_name="adk-eval-loop-cookbook", project_type=ProjectType.OBSERVE,
                    set_global_tracer_provider=True, verbose=False)
GoogleADKInstrumentor().instrument(tracer_provider=provider)
enable_auto_enrichment()
tracer = provider.get_tracer("weather-cookbook")

## Give the agent a known source of truth

In [ ]:
from uuid import uuid4, UUID
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

QUESTION = "What is the weather in New York? Use the weather tool."
REPORT = "The weather in New York is sunny with a temperature of 25 degrees Celsius."


def get_weather(city: str) -> dict:
    """Look up the synthetic weather report for a city."""
    if city.strip().casefold() in {"new york", "new york city", "nyc"}:
        return {"status": "success", "report": REPORT}
    return {"status": "unavailable"}


def weather_agent(model):
    return Agent(name="weather_agent", model=model, tools=[get_weather],
                 instruction="Always call get_weather. Answer only from its result.")


async def run_once(model):
    app = "weather_cookbook"
    runner = InMemoryRunner(agent=weather_agent(model), app_name=app)
    try:
        session = await runner.session_service.create_session(
            app_name=app, user_id="demo-hiker", session_id=uuid4().hex)
        message = types.Content(role="user", parts=[types.Part(text=QUESTION)])
        tools, reply = [], ""
        async for event in runner.run_async(user_id="demo-hiker", session_id=session.id,
                                           new_message=message):
            tools.extend(call.name for call in event.get_function_calls())
            if event.is_final_response() and event.content:
                reply = "".join(p.text or "" for p in event.content.parts or []).strip()
        if "get_weather" not in tools or not reply:
            raise RuntimeError("The agent did not both call the tool and give a final answer.")
        return reply
    finally:
        await runner.close()

## Score the answer inside its trace

Use top-level `await` in Jupyter. The parent span stays open while ADK runs and while the kit creates its `eval.groundedness` span.

In [ ]:
async def score_turn(model):
    with tracer.start_as_current_span("weather_agent_groundedness_check") as span:
        trace_id = str(UUID(int=span.get_span_context().trace_id))
        answer = await run_once(model)
        verdict = evaluate("groundedness", input=QUESTION, output=answer,
                           context=REPORT, model="turing_small")
        if verdict.status != "completed" or verdict.score is None or verdict.passed is None:
            raise RuntimeError(f"Groundedness did not return a verdict: {verdict.error or verdict.reason}")
    if not provider.force_flush():
        raise RuntimeError("The trace exporter did not flush in time.")
    return trace_id, answer, verdict


trace_id, answer, verdict = await score_turn(LiteLlm(model="openai/gpt-4o-mini"))
print("Trace ID:", trace_id)
print("Reply:", answer)
print("Groundedness:", verdict.score, verdict.reason)

## Read back the judgment and apply a gate

Use the saved ID rather than looking for the latest run. In **Observe → adk-eval-loop-cookbook**, open the displayed trace ID to see the agent and evaluation together.

In [ ]:
eval_id = verdict.metadata["eval_id"]
stored = Evaluator().get_eval_result(eval_id)["result"]["result"]
if str(stored["eval_id"]) != str(eval_id) or stored["name"] != "groundedness" or stored.get("error_message"):
    raise RuntimeError("Stored evaluation does not match this turn.")
print("Gate:", "PASS" if verdict.passed else "REVIEW")

## Optional: use Gemini instead

For the Gemini Developer API, set `GOOGLE_API_KEY`. For Vertex AI instead, configure Application Default Credentials, `GOOGLE_GENAI_USE_VERTEXAI=true`, `GOOGLE_CLOUD_PROJECT` and `GOOGLE_CLOUD_LOCATION`. Run this cell manually when you have the corresponding access; it makes another model call and a new judgment.

In [ ]:
using_vertex = os.getenv("GOOGLE_GENAI_USE_VERTEXAI", "").lower() in {"1", "true"}
if os.getenv("GOOGLE_API_KEY") or using_vertex:
    gemini_trace, gemini_reply, gemini_result = await score_turn("gemini-2.5-flash")
    print("Gemini:", gemini_trace, gemini_reply, gemini_result.score)
else:
    print("Optional: configure Gemini or Vertex AI before running this variation.")

## Next steps

Build a regression suite with additional cities, including one the tool does not know. Treat tool use and groundedness as separate checks: a fluent answer can fail either. See [Evaluation-driven development](https://docs.futureagi.com/docs/cookbook/quickstart/eval-driven-dev) for a CI gate and [Prompt optimization](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-optimization) for a scored revision loop.